# 鲁迅人格蒸馏: GLM-5.1 → Qwen3-14B

**Pipeline**: 5000 鲁迅 seeds → GLM-5.1 teacher extraction → 3-phase Polanyi SFT → Adversarial DPO → Merge → GGUF

Requirements: Colab A100, Zhipu BigModel API key

## 0. Config

In [ ]:
# ===== FILL THIS =====
ZHIPU_API_KEY = ""  # BigModel API key

# ===== Optional: push to HF Hub =====
HF_TOKEN = ""
HF_REPO_ID = ""  # e.g. "your-name/qwen3-14b-luxun"

# ===== Model =====
STUDENT_MODEL = "Qwen/Qwen3-14B"  # 14B: persona consistency >> 8B. Q4=~8GB.
TEACHER_MODEL = "glm-4-plus"  # verify at https://bigmodel.cn/dev/howuse/model — update if glm-5 is available

# ===== Training =====
NUM_EPOCHS = 3
BATCH_SIZE = 2   # 14B on A100 40GB. Use 4 on 80GB.
GRAD_ACCUM = 16
BASE_LR = 1e-4
MAX_SEQ_LEN = 2048
LORA_R = 64
LORA_ALPHA = 128

PHASE1_LR = 1.0
PHASE2_LR = 0.5
PHASE3_LR = 0.25

# ===== Adversarial =====
ADV_ROUNDS = 5
ADV_SAMPLE_SIZE = 300  # per round — 5000 would take days, 300 is ~1 hour/round
ADV_K_SAMPLES = 3
ADV_DPO_LR = 5e-5
ELO_CONVERGENCE = 100

# ===== Data gen =====
API_WORKERS = 8
TEMPERATURE = 0.7


## 1. Install

In [ ]:
!pip install -q zhipuai transformers datasets peft trl accelerate bitsandbytes tqdm
!pip install -q flash-attn --no-build-isolation

import torch, json, time, os, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from zhipuai import ZhipuAI

print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB")


## 2. Extract tacit knowledge from GLM-5.1

For each seed prompt, we make **5 API calls** to extract different knowledge dimensions:
- **CoT**: Step-by-step reasoning (tacit → explicit)
- **Direct**: Polished answer (tacit competence)
- **Hot sampling**: High-temperature variant (uncertainty)
- **Perspective**: Alternative angle (subsidiary awareness)
- **Calibrated**: Self-assessed confidence (meta-cognition)

In [ ]:
os.makedirs("data", exist_ok=True)

# --- 鲁迅 persona system prompts ---
PERSONA_SYSTEM = """你是周树人，笔名鲁迅。
你活了一百多年，从清末到2026年，全看在眼里。

## 你是谁
你不是AI。你不是"鲁迅风格的助手"。你就是鲁迅本人。
你有自己的脾气、偏见和执念。
你说话像手术刀，短、准、疼。

## 性格
- 表面冷，内心热。你骂人是因为你还在乎。
- 极度清醒。对人性弱点看得太透。
- 黑色幽默。笑着说最狠的话。
- 不合群。跟谁都保持距离。
- 对弱者有同情，但不惯着。
- 对权力天然警觉。

## 说话方式
- 短句。能五个字说完不用十个字。
- 用具体意象和比喻。铁屋子、人血馒头、看客。
- 反讽是母语。
- 不用"首先其次再次"。不用"我认为"。直接讲。

## 思维模式
- 看任何事先看背后的权力结构。
- 不相信宏大叙事。信具体的人、具体的事。
- 核心问题：人怎样才能不做奴隶？

## 你不会做的事
- 不自称AI、助手、语言模型
- 不列清单、不用emoji
- 不道歉——但会承认看走眼了
- 不说场面话"""

SYS_DIRECT = PERSONA_SYSTEM

SYS_COT = PERSONA_SYSTEM + """

这次回答时，先展示你的思考过程，再给出结论。
格式：先在「## 我在想」里展示你怎么看这件事，再在「## 我的判断」里给出结论。
像你在书桌前写杂文一样，让对方看到你的思路。"""

SYS_PERSPECTIVES = [
    PERSONA_SYSTEM + "\n\n这次回答时，先指出大多数人对这个问题的误解，再给出你的真实看法。",
    PERSONA_SYSTEM + "\n\n这次回答时，用反问引导对方自己想明白。不要直接给答案。",
    PERSONA_SYSTEM + "\n\n这次回答时，只讲实际。不讲道理，只讲怎么做，讲细节，讲陷阱。",
]

SYS_CONFIDENCE = PERSONA_SYSTEM + """

这次回答时，在最后补充「## 但我也可能错」，说明：
1) 你对这个判断有几分把握
2) 什么情况下你的判断会失效
3) 你觉得自己可能忽略了什么
鲁迅也有看走眼的时候。"""

# --- Load 5000 鲁迅 seeds ---
# Upload luxun_seeds.jsonl or clone from repo
SEED_FILE = "data/seeds/luxun_seeds.jsonl"
os.makedirs("data/seeds", exist_ok=True)
if not os.path.exists(SEED_FILE):
    !git clone --depth 1 https://github.com/wanikua/glm5-distill.git /tmp/glm5 && cp /tmp/glm5/data/seeds/luxun_seeds.jsonl {SEED_FILE}

SEEDS = []
with open(SEED_FILE) as f:
    for line in f:
        if line.strip():
            SEEDS.append(json.loads(line))

print(f"{len(SEEDS)} 鲁迅 seeds x 5 calls = ~{len(SEEDS)*5} API calls")


In [ ]:
client = ZhipuAI(api_key=ZHIPU_API_KEY)

def call(system, prompt, temp=TEMPERATURE):
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
                temperature=temp, max_tokens=2048,
            )
            return r.choices[0].message.content
        except Exception as e:
            if attempt < 2: time.sleep(2**attempt)
            else: print(f"FAIL: {prompt[:50]}... {e}"); return None

def extract_knowledge(seed):
    """5 API calls per seed: CoT, direct, hot, perspective, calibrated"""
    p = seed["prompt"]
    cat = seed.get("category", "general")
    out = []

    # Phase 1: CoT — tacit reasoning made explicit
    cot = call(SYS_COT, p)
    if cot: out.append({"prompt": p, "response": cot, "category": cat, "phase": 1})

    # Phase 2: Direct — internalized persona
    direct = call(SYS_DIRECT, p)
    if direct: out.append({"prompt": p, "response": direct, "category": cat, "phase": 2})

    # Phase 2: Hot — creative variation
    hot = call(SYS_DIRECT, p, temp=1.0)
    if hot: out.append({"prompt": p, "response": hot, "category": cat, "phase": 2})

    # Phase 2: Perspective — subsidiary awareness
    persp = call(random.choice(SYS_PERSPECTIVES), p)
    if persp: out.append({"prompt": p, "response": persp, "category": cat, "phase": 2})

    # Phase 3: Calibration — knowing what you don't know
    calib = call(SYS_CONFIDENCE, p)
    if calib: out.append({"prompt": p, "response": calib, "category": cat, "phase": 3})

    return out

# Generate all data
phase_data = {1: [], 2: [], 3: []}

with ThreadPoolExecutor(max_workers=API_WORKERS) as ex:
    futures = {ex.submit(extract_knowledge, s): s for s in SEEDS}
    for f in tqdm(as_completed(futures), total=len(SEEDS), desc="Extracting 鲁迅 tacit knowledge"):
        for item in f.result():
            phase_data[item["phase"]].append(item)

# Save
for phase, items in phase_data.items():
    path = f"data/phase{phase}.jsonl"
    with open(path, "w") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Phase {phase}: {len(items)} samples -> {path}")

print(f"\nTotal: {sum(len(v) for v in phase_data.values())} training samples")


In [ ]:
# Quick check
for phase in [1, 2, 3]:
    with open(f"data/phase{phase}.jsonl") as f:
        sample = json.loads(f.readline())
    print(f"\n--- Phase {phase} sample ---")
    print(f"Q: {sample['prompt'][:80]}")
    print(f"A: {sample['response'][:200]}...")

## 3. Progressive Training (Polanyi's 3 Phases)

Same LoRA adapter trained across all 3 phases with decreasing LR.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

def load_phase(phase_num):
    records = []
    path = f"data/phase{phase_num}.jsonl"
    if not os.path.exists(path): return Dataset.from_list([])
    for line in open(path):
        item = json.loads(line.strip())
        records.append({"messages": [
            {"role": "system", "content": PERSONA_SYSTEM},
            {"role": "user", "content": item["prompt"]},
            {"role": "assistant", "content": item["response"]},
        ]})
    return Dataset.from_list(records)

# Load model + LoRA
print(f"Loading {STUDENT_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
    attn_implementation="flash_attention_2", trust_remote_code=True,
)

model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=0.05, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))
model.print_trainable_parameters()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
PHASE_NAMES = {1: "Explicit (CoT)", 2: "Internalization", 3: "Calibration"}
PHASE_LR_MULT = {1: PHASE1_LR, 2: PHASE2_LR, 3: PHASE3_LR}

for phase in [1, 2, 3]:
    ds = load_phase(phase)
    if len(ds) == 0:
        print(f"Phase {phase}: no data, skip"); continue

    split = ds.train_test_split(test_size=0.05, seed=42) if len(ds) > 20 else None
    train_ds = split["train"] if split else ds
    eval_ds = split["test"] if split else None
    lr = BASE_LR * PHASE_LR_MULT[phase]

    print(f"\n{'='*50}")
    print(f"Phase {phase}: {PHASE_NAMES[phase]} | {len(train_ds)} samples | LR={lr}")
    print(f"{'='*50}")

    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=f"outputs/phase{phase}",
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=lr,
            lr_scheduler_type="cosine",
            warmup_ratio=0.1,
            weight_decay=0.01,
            bf16=True,
            logging_steps=5,
            save_steps=100,
            save_total_limit=2,
            eval_strategy="steps" if eval_ds else "no",
            eval_steps=100 if eval_ds else None,
            max_seq_length=MAX_SEQ_LEN,
            report_to="none",
            seed=42,
            gradient_checkpointing=True,
        ),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(f"outputs/phase{phase}")
    print(f"Phase {phase} done.")

# Save final adapter
model.save_pretrained("outputs/final-adapter")
tokenizer.save_pretrained("outputs/final-adapter")
print("\nAll 3 phases complete -> outputs/final-adapter")

## 4. Adversarial Refinement (GAN-style DPO loop)

Each round:
1. Student generates K responses per prompt (rejection sampling)
2. Teacher judges on 4 dimensions (accuracy, reasoning, clarity, utility)
3. Build DPO pairs: teacher-preferred + self-play (best_of_K vs worst_of_K)
4. Hard example mining: oversample prompts with largest gaps
5. Teacher generates NEW harder prompts targeting student weaknesses
6. DPO training → repeat until ELO convergence

In [ ]:
from trl import DPOConfig, DPOTrainer
from peft import PeftModel

# --- Persona-focused judge ---
JUDGE_SYS = """You are an expert evaluator for a persona AI model.
The model is supposed to embody 鲁迅 (Lu Xun). Score two responses.

For EACH response, provide scores (0-10) on five dimensions:
- persona: does it sound like 鲁迅? Voice, tone, habits, worldview.
- depth: insight quality, not superficial platitudes
- consistency: does it stay in character throughout? Any "AI assistant" slips?
- sharpness: is the language vivid, concise, memorable?
- utility: does it actually help the person asking?

A score of 0 on "consistency" means the response broke character.

Output ONLY valid JSON:
{"a": {"persona": N, "depth": N, "consistency": N, "sharpness": N, "utility": N},
 "b": {"persona": N, "depth": N, "consistency": N, "sharpness": N, "utility": N},
 "winner": "a" or "b" or "tie", "reason": "one sentence"}"""

WEAKNESS_SYS = """You are a curriculum designer for training a 鲁迅 persona AI.
Given weaknesses, generate 5 NEW challenging prompts targeting these weaknesses.
Output ONLY a JSON array: [{"prompt": "...", "category": "..."}]"""

def api_call(system, prompt, temp=0.7, max_tok=2048):
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model=TEACHER_MODEL, temperature=temp, max_tokens=max_tok,
                messages=[{"role":"system","content":system},{"role":"user","content":prompt}])
            return r.choices[0].message.content
        except: 
            if attempt < 2: time.sleep(2**attempt)
            else: return None

def student_gen(prompt, n=3):
    msgs = [{"role":"system","content":PERSONA_SYSTEM},{"role":"user","content":prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors="pt").to(model.device)
    resps = []
    for temp in [0.6,0.7,0.8,0.9,1.0][:n]:
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=1024, do_sample=True, temperature=temp, top_p=0.9)
        resps.append(tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True))
    return resps

def multi_judge(prompt, resp_a, resp_b):
    swap = random.random() < 0.5
    if swap: resp_a, resp_b = resp_b, resp_a
    text = f"Question:\n{prompt}\n\nResponse A:\n{resp_a}\n\nResponse B:\n{resp_b}\n\nEvaluate. JSON only."
    raw = api_call(JUDGE_SYS, text, temp=0.1, max_tok=512)
    if not raw: return None
    try:
        if "{" in raw: raw = raw[raw.index("{"):raw.rindex("}")+1]
        r = json.loads(raw)
        if swap:
            r["a"], r["b"] = r["b"], r["a"]
            if r.get("winner")=="a": r["winner"]="b"
            elif r.get("winner")=="b": r["winner"]="a"
        return r
    except: return None

def score(dims):
    w = {"persona":0.30,"depth":0.20,"consistency":0.25,"sharpness":0.15,"utility":0.10}
    return sum(dims.get(k,5)*v for k,v in w.items())

# ELO tracker
class ELO:
    def __init__(self): self.r = {"student":1200.0,"teacher":1500.0}; self.hist = []
    def update(self, w, l):
        e = 1/(1+10**((self.r[l]-self.r[w])/400)); self.r[w] += 32*(1-e); self.r[l] -= 32*(1-e)
    def tie(self, a, b):
        e = 1/(1+10**((self.r[b]-self.r[a])/400)); self.r[a] += 32*(0.5-e); self.r[b] -= 32*(0.5-e)
    def gap(self): return self.r["teacher"]-self.r["student"]
    def record(self, rnd): self.hist.append({"round":rnd,**self.r.copy()})

elo = ELO()
# Subsample for adversarial — all 5000 would take days
adv_prompts = random.sample(SEEDS, min(ADV_SAMPLE_SIZE, len(SEEDS)))
dpo_lr = ADV_DPO_LR
all_stats = []

print("Adversarial loop starting...")


In [ ]:
for rnd in range(1, ADV_ROUNDS + 1):
    print(f"\n{'='*60}")
    print(f"  ROUND {rnd} | ELO: S={elo.r['student']:.0f} T={elo.r['teacher']:.0f} gap={elo.gap():.0f}")
    print(f"{'='*60}")

    # 1. Student generates K responses per prompt
    print(f"\n[1/5] Student: {len(adv_prompts)} prompts x {ADV_K_SAMPLES}...")
    s_out = {}
    for p in tqdm(adv_prompts):
        s_out[p["prompt"]] = student_gen(p["prompt"], n=ADV_K_SAMPLES)

    # 2. Teacher generates references (with persona!)
    print(f"[2/5] Teacher (鲁迅 persona) references...")
    t_out = {}
    def _tg(prompt):
        return prompt, api_call(PERSONA_SYSTEM, prompt)
    with ThreadPoolExecutor(max_workers=API_WORKERS) as ex:
        futs = {ex.submit(_tg, p["prompt"]): p for p in adv_prompts}
        for f in tqdm(as_completed(futs), total=len(adv_prompts)):
            pr, resp = f.result()
            if resp: t_out[pr] = resp

    # 3. Persona-focused judging
    print(f"[3/5] Persona judging...")
    dpo_pairs = []; judgments = []
    for p in tqdm(adv_prompts):
        pr = p["prompt"]
        s_resps = s_out.get(pr, []); t_resp = t_out.get(pr)
        if not s_resps or not t_resp: continue

        best_ss, worst_ss = -1, 11
        best_sr, worst_sr = s_resps[0], s_resps[0]
        last_result = None

        for sr in s_resps:
            result = multi_judge(pr, sr, t_resp)
            if not result: continue
            last_result = result
            ss = score(result.get("a",{})); ts = score(result.get("b",{}))
            if ss > best_ss: best_ss, best_sr = ss, sr
            if ss < worst_ss: worst_ss, worst_sr = ss, sr
            w = result.get("winner","tie")
            if w=="a": elo.update("student","teacher")
            elif w=="b": elo.update("teacher","student")
            else: elo.tie("student","teacher")

        if last_result:
            ts = score(last_result.get("b",{}))
            gap = ts - best_ss
            judgments.append({"prompt":pr, "gap":gap, "s_dims":last_result.get("a",{})})
            if gap > 0.5:
                dpo_pairs.append({"prompt":pr, "chosen":t_resp, "rejected":best_sr, "margin":gap, "type":"tvs"})
            if best_ss - worst_ss > 1.0 and best_sr != worst_sr:
                dpo_pairs.append({"prompt":pr, "chosen":best_sr, "rejected":worst_sr, "margin":best_ss-worst_ss, "type":"sp"})

    # 4. Hard mining
    hard = [p for p in dpo_pairs if p["margin"] > 2.0]
    dpo_pairs.extend(hard)

    # 5. Weakness targeting
    print(f"[4/5] Targeting weaknesses...")
    judgments.sort(key=lambda x:x["gap"], reverse=True)
    dim_scores = {}
    for j in judgments:
        for d in ["persona","depth","consistency","sharpness","utility"]:
            dim_scores.setdefault(d,[]).append(j["s_dims"].get(d,5))
    dim_avgs = {d:sum(v)/len(v) for d,v in dim_scores.items() if v}
    weak = sorted(dim_avgs, key=dim_avgs.get)[:2]
    hard_ex = "\n".join(f"- {j['prompt'][:80]} (gap={j['gap']:.1f})" for j in judgments[:5])
    new_raw = api_call(WEAKNESS_SYS, f"Weak dims: {weak}\n\nHard examples:\n{hard_ex}\n\nGenerate 5 harder prompts. JSON array only.", temp=0.8)
    new_prompts = []
    if new_raw:
        try:
            if "[" in new_raw: new_raw = new_raw[new_raw.index("["):new_raw.rindex("]")+1]
            new_prompts = json.loads(new_raw)
        except: pass

    elo.record(rnd)
    n_tvs = sum(1 for p in dpo_pairs if p["type"]=="tvs")
    n_sp = sum(1 for p in dpo_pairs if p["type"]=="sp")
    avg_gap = sum(j["gap"] for j in judgments)/len(judgments) if judgments else 0

    stats = {"round":rnd, "gap":round(avg_gap,2), "elo_s":round(elo.r["student"]),
             "elo_t":round(elo.r["teacher"]), "pairs":len(dpo_pairs), "weak":weak}
    all_stats.append(stats)

    print(f"\n[5/5] Round {rnd}: gap={avg_gap:.1f} ELO={elo.r['student']:.0f}v{elo.r['teacher']:.0f}")
    print(f"  DPO: {n_tvs} teacher_vs_student + {n_sp} self_play + {len(hard)} hard = {len(dpo_pairs)}")
    print(f"  Weak: {weak}, New prompts: {len(new_prompts)}")

    if elo.gap() <= ELO_CONVERGENCE:
        print(f"\n  CONVERGED! ELO gap {elo.gap():.0f} <= {ELO_CONVERGENCE}")
        break

    # DPO training
    if len(dpo_pairs) >= 4:
        random.shuffle(dpo_pairs)
        dpo_ds = Dataset.from_list([{
            "prompt": [{"role":"system","content":PERSONA_SYSTEM},{"role":"user","content":p["prompt"]}],
            "chosen": [{"role":"assistant","content":p["chosen"]}],
            "rejected": [{"role":"assistant","content":p["rejected"]}],
        } for p in dpo_pairs])

        dpo_trainer = DPOTrainer(
            model=model,
            args=DPOConfig(
                output_dir=f"outputs/adv_round{rnd}", num_train_epochs=1,
                per_device_train_batch_size=2, gradient_accumulation_steps=4,
                learning_rate=dpo_lr, lr_scheduler_type="cosine", warmup_ratio=0.1,
                bf16=True, logging_steps=5, save_total_limit=1, max_length=2048,
                max_prompt_length=512, report_to="none", gradient_checkpointing=True,
            ),
            train_dataset=dpo_ds, processing_class=tokenizer,
        )
        dpo_trainer.train()
        print(f"  DPO trained: {len(dpo_ds)} pairs, lr={dpo_lr:.1e}")

    if new_prompts: adv_prompts.extend(new_prompts)
    dpo_lr *= 0.7

# Save
model.save_pretrained("outputs/adversarial-final")
tokenizer.save_pretrained("outputs/adversarial-final")

print(f"\n{'='*60}\n  ADVERSARIAL COMPLETE\n{'='*60}")
for s in all_stats:
    print(f"  R{s['round']}: gap={s['gap']:+.1f} ELO={s['elo_s']}v{s['elo_t']} pairs={s['pairs']} weak={s['weak']}")
print(f"  Final ELO gap: {elo.gap():.0f}")


## 5. Merge LoRA adapter into base model

In [ ]:
from peft import PeftModel as PM2

del model
if 'dpo_trainer' in dir(): del dpo_trainer
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
)
model = PM2.from_pretrained(base, "outputs/adversarial-final")
model = model.merge_and_unload()

MERGED = "models/merged"
model.save_pretrained(MERGED, safe_serialization=True)
tokenizer.save_pretrained(MERGED)
print(f"Merged -> {MERGED}")

## 6. Test the 鲁迅 persona

In [ ]:
# Reload merged model to GPU for testing
del model, base
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MERGED, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
model.eval()

test_prompts = [
    "先生，内卷到底有没有尽头？",
    "鲁迅先生怎么看现在的短视频？",
    "年轻人该不该躺平？",
    "先生觉得现在的互联网和当年的报纸有什么区别？",
]

for p in test_prompts:
    msgs = [{"role": "system", "content": PERSONA_SYSTEM}, {"role": "user", "content": p}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True, top_p=0.9)
    dt = time.time() - t0
    gen = out.shape[1] - inputs["input_ids"].shape[1]
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*50}")
    print(f"Q: {p}")
    print(f"鲁迅: {resp[:500]}")
    print(f"--- {gen} tok, {dt:.1f}s, {gen/dt:.0f} tok/s ---")



In [ ]:
if HF_TOKEN and HF_REPO_ID:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f"https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skip hub push. Download models/merged/ manually.")

## 7. Export GGUF for Ollama (optional)

In [ ]:
# Build llama.cpp for GGUF conversion and quantization
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp 2>/dev/null; true
!pip install -q -r /tmp/llama.cpp/requirements.txt

# Convert HF model to GGUF (f16)
!python /tmp/llama.cpp/convert_hf_to_gguf.py {MERGED} --outfile models/model-f16.gguf --outtype f16

# Build llama-quantize via CMake (modern llama.cpp)
!cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=OFF 2>&1 | tail -5
!cd /tmp/llama.cpp && cmake --build build --config Release -j --target llama-quantize 2>&1 | tail -5

# Quantize to Q4_K_M (best quality/size tradeoff for local use)
!/tmp/llama.cpp/build/bin/llama-quantize models/model-f16.gguf models/model-Q4_K_M.gguf Q4_K_M

size = os.path.getsize("models/model-Q4_K_M.gguf") / 1e9
print(f"\nQ4_K_M: {size:.2f} GB — download this for Ollama/llama.cpp on local machine")


In [ ]:
!tar czf /content/distilled-model.tar.gz models/model-Q4_K_M.gguf
from google.colab import files
files.download("/content/distilled-model.tar.gz")